# MIP Logistic Regression to sklearn

Use the high-level `FederatedLogisticRegression` class with a transient experiment JSON payload.
It runs only `logistic_regression` through platform-backend and returns a ready-to-use sklearn model.

In [1]:
import numpy as np
import pandas as pd
import joblib
from platform_backend_client import configure, FederatedLogisticRegression

configure()


In [2]:
# Fill this JSON payload with your transient experiment input.
# Only logistic_regression is supported by FederatedLogisticRegression.
payload = {
  "name": "experiment_logistic_regression",
  "algorithm": {
    "name": "logistic_regression",
    "inputdata": {
      "data_model": "dementia:0.1",
      "y": [
        "agegroup"
      ],
      "x": [
        "subjectageyears"
      ],
      "datasets": [
        "ppmi",
        "edsd",
        "desd-synthdata"
      ],
      "filters": None
    },
    "parameters": {
      "positive_class": "50-59y"
    },
    "preprocessing": None
  },
  "mipVersion": "9.0.0"
}
payload


{'name': 'experiment_logistic_regression',
 'algorithm': {'name': 'logistic_regression',
  'inputdata': {'data_model': 'dementia:0.1',
   'y': ['agegroup'],
   'x': ['subjectageyears'],
   'datasets': ['ppmi', 'edsd', 'desd-synthdata'],
   'filters': None},
  'parameters': {'positive_class': '50-59y'},
  'preprocessing': None},
 'mipVersion': '9.0.0'}

In [3]:
model = FederatedLogisticRegression(payload)


print("experiment uuid:", model.experiment_uuid)
print("status:", model.status)
print("classes_:", model.classes_)
print("coef_.shape:", model.coef_.shape)
print("intercept_:", model.intercept_)


experiment uuid: 3e985132-eab7-4136-a2df-d78ae50341cd
status: success
classes_: [0 1]
coef_.shape: (1, 1)
intercept_: [6.03726621]


In [4]:
x_new = np.zeros((3, model.n_features_in_), dtype=float)
if model.n_features_in_ >= 1:
    x_new[1, 0] = 1.0
if model.n_features_in_ >= 2:
    x_new[2, 1] = 1.0

feature_names = getattr(model, "feature_names_in_", [f"x{i}" for i in range(model.n_features_in_)])
x_new_df = pd.DataFrame(x_new, columns=list(feature_names))

print("predict:", model.predict(x_new_df))
print("predict_proba:")
model.predict_proba(x_new_df)


predict: [1 1 1]
predict_proba:


array([[0.00238239, 0.99761761],
       [0.00270041, 0.99729959],
       [0.00238239, 0.99761761]])

In [10]:
model_path = "logistic_regression_from_platform.joblib"
model.dump(model_path)
reloaded_model = joblib.load(model_path)
print("saved and reloaded model from:", model_path)
print("reloaded coef_.shape:", reloaded_model.coef_.shape)


saved and reloaded model from: logistic_regression_from_platform.joblib
reloaded coef_.shape: (1, 2)


You can also initialize from a JSON string with `FederatedLogisticRegression.from_json(json_string)`.